* **Load and process Square CSV export into file suitable for upload to RENXT**
* <code>square2renxt.ipynb</code>
* Mark F. Russo
* v. 20261109

In [ ]:
import pandas as pd

# Load mock Square data export as CSV file and display info.
df = pd.read_csv('square_transactions_large_multi_line.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Date             520 non-null    str  
 1   Time             520 non-null    str  
 2   Transaction ID   520 non-null    str  
 3   Item             520 non-null    str  
 4   SKU              520 non-null    str  
 5   Category         520 non-null    str  
 6   Quantity         520 non-null    int64
 7   Gross Sales      520 non-null    str  
 8   Discounts        520 non-null    str  
 9   Net Sales        520 non-null    str  
 10  Tax              520 non-null    str  
 11  Total Collected  520 non-null    str  
 12  Payment Method   520 non-null    str  
 13  Device Name      520 non-null    str  
dtypes: int64(1), str(13)
memory usage: 105.6 KB


In [46]:
# The columns Gross Sales, Discounts, Net Sales, Tax, Total Collected are not numeric due to the $ and comma characters.
# Strip extra characters and convert data in these columns to cents (integer numbers).
# By using integers we avoid problems floating point number rounding and comparison.
# We can convert back to dollars before import. 

# Back up string currency columns as new columns in DataFrame
df['Gross Sales (orig)'] = df['Gross Sales']
df['Discounts (orig)'] = df['Discounts']
df['Net Sales (orig)'] = df['Net Sales']
df['Tax (orig)'] = df['Tax']
df['Total Collected (orig)'] = df['Total Collected']

# Strip all dollars, decimals, and commas from strings, and then change to int64 numeric types and add a new column.
df['Gross Sales'] = pd.to_numeric(df['Gross Sales'].str.replace(r'[^\d-]', '', regex=True), errors='coerce')
df['Discounts'] = pd.to_numeric(df['Discounts'].str.replace(r'[^\d-]', '', regex=True), errors='coerce')
df['Net Sales'] = pd.to_numeric(df['Net Sales'].str.replace(r'[^\d-]', '', regex=True), errors='coerce')
df['Tax'] = pd.to_numeric(df['Tax'].str.replace(r'[^\d-]', '', regex=True), errors='coerce')
df['Total Collected'] = pd.to_numeric(df['Total Collected'].str.replace(r'[^\d-]', '', regex=True), errors='coerce')

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Date                    520 non-null    str  
 1   Time                    520 non-null    str  
 2   Transaction ID          520 non-null    str  
 3   Item                    520 non-null    str  
 4   SKU                     520 non-null    str  
 5   Category                520 non-null    str  
 6   Quantity                520 non-null    int64
 7   Gross Sales             520 non-null    int64
 8   Discounts               520 non-null    int64
 9   Net Sales               520 non-null    int64
 10  Tax                     520 non-null    int64
 11  Total Collected         520 non-null    int64
 12  Payment Method          520 non-null    str  
 13  Device Name             520 non-null    str  
 14  Gross Sales (orig)      520 non-null    str  
 15  Discounts (orig)        520 non-nu

In [48]:
# List of unique values in the Category column of mock data set.
df['Category'].unique()

<ArrowStringArray>
['Bakery', 'Tea & Other', 'Merchandise', 'Coffee']
Length: 4, dtype: str

In [49]:
# [Nick: Per transaction, classifies each line item by type (either contribution or exchange).]

# Add a new column to DataFrame titled "Class" that arbitrarily classifies 
# the categories 'Bakery', 'Tea & Other', and 'Coffee', as 'exchange' and 
# the category 'Merchandise' as 'contribution'.
# These can be changed to meaningful values.

# Mapping
map = {'Bakery':'exchange', 'Tea & Other':'exchange', 'Coffee':'exchange', 'Merchandise':'contribution'}

# Apply map and add mapped data as new Column in DataFrame named 'Class'
df['Class'] = df['Category'].map( map )
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Date                    520 non-null    str  
 1   Time                    520 non-null    str  
 2   Transaction ID          520 non-null    str  
 3   Item                    520 non-null    str  
 4   SKU                     520 non-null    str  
 5   Category                520 non-null    str  
 6   Quantity                520 non-null    int64
 7   Gross Sales             520 non-null    int64
 8   Discounts               520 non-null    int64
 9   Net Sales               520 non-null    int64
 10  Tax                     520 non-null    int64
 11  Total Collected         520 non-null    int64
 12  Payment Method          520 non-null    str  
 13  Device Name             520 non-null    str  
 14  Gross Sales (orig)      520 non-null    str  
 15  Discounts (orig)        520 non-nu

In [50]:
# [Nick: Per transaction, sums line items of the same type to produce a subtotal for each.]
# [Nick: Per subtotal, records the original Square transaction ID to which the subtotal belongs (for audit purposes)]

# Produce a new DataFrame that groups by transaction id and assigned Class and sums Total Collected.
grouped = df[['Transaction ID','Total Collected','Class']].groupby(['Transaction ID','Class'], as_index=False).sum()
grouped

,Transaction ID,Class,Total Collected
0,TX-5001,exchange,974
1,TX-5002,contribution,1949
2,TX-5002,exchange,622
3,TX-5003,contribution,1949
4,TX-5004,exchange,1786
...,...,...,...
302,TX-5206,exchange,1407
303,TX-5207,exchange,812
304,TX-5208,exchange,1596
305,TX-5209,exchange,893


In [51]:
# [Nick: Confirms that the subtotals match the original Square transaction total.]

# Regroup new DataFrame and original DataFrame by Transaction ID only and sum Total Collected.
# Match two DataFrames to ensure they are the same.
tot1 = grouped.groupby('Transaction ID')['Total Collected'].sum()
tot2 = df.groupby('Transaction ID')['Total Collected'].sum()
print(tot1.equals(tot2))

# The following shows where DataFrame values are different, if any
tot1.compare(tot2)

True


,self,other
Transaction ID,,


In [52]:
# Finally, convert back to dollars
grouped['Total Collected'] = grouped['Total Collected']/100
grouped

,Transaction ID,Class,Total Collected
0,TX-5001,exchange,9.74
1,TX-5002,contribution,19.49
2,TX-5002,exchange,6.22
3,TX-5003,contribution,19.49
4,TX-5004,exchange,17.86
...,...,...,...
302,TX-5206,exchange,14.07
303,TX-5207,exchange,8.12
304,TX-5208,exchange,15.96
305,TX-5209,exchange,8.93


In [53]:
# Export to new CSV file
grouped.to_csv('RENXT.csv')